In [8]:
!pip install qiskit qiskit-aer --quiet

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer


In [9]:
program = [
    "00000000",
    "01010000",
    "10000100",
    "11011001",
    "11000000"
]

opcode_map = {
    "00": "H",
    "01": "X",
    "10": "CX",
    "11": "MAC/MEASURE"
}


In [10]:
class QRISCProcessor:
    def __init__(self):
        self.qc = QuantumCircuit(8, 8)
        self.pc = 0

        self.reg = [0]*8
        self.last_used = [-1]*8

        self.Z = 0
        self.P = 0
        self.C = 0

    # Fecth wala part
    def fetch(self):
        if self.pc < len(program):
            instr = program[self.pc]
            print(f"\n[FETCH] PC={self.pc} → {instr}")
            self.pc += 1
            return instr
        return None

    # Decode wala part
    def decode(self, instr):
        opcode = instr[0:2]
        q1 = int(instr[2:4], 2)
        q2 = int(instr[4:6], 2)
        q3 = int(instr[6:8], 2)

        op = opcode_map.get(opcode, "UNKNOWN")

        print(f"[DECODE] {op} | q1={q1}, q2={q2}, q3={q3}")
        return op, q1, q2, q3


    def hazard_check(self, q1, q2):
        if self.last_used[q1] == self.pc-1 or self.last_used[q2] == self.pc-1:
            print("Hazard detected so do Stalling of pipeline")


    def ensure_connectivity(self, q1, q2):
        if abs(q1 - q2) > 1:
            print("Connectivity issue so do Applying of SWAPs")
            for i in range(min(q1, q2), max(q1, q2)):
                self.qc.swap(i, i+1)


    def update_flags(self, value):
        self.Z = 1 if value == 0 else 0
        self.P = value % 2


    def execute(self, op, q1, q2, q3):

        self.hazard_check(q1, q2)

        print(f"[EXECUTE] {op}")

        if op == "H":
            self.qc.h(q1)

        elif op == "X":
            self.qc.x(q1)
            self.reg[q1] ^= 1
            self.update_flags(self.reg[q1])

        elif op == "CX":
            self.ensure_connectivity(q1, q2)
            self.qc.cx(q1, q2)

        elif op == "MAC/MEASURE":

            if q2 == 0 and q3 == 0:
                self.qc.measure(q1, q1)
                print(f"Measured q{q1}")

                if self.reg[q1] == 1:
                    print("Branch taken so do skipping of next instruction")
                    self.pc += 1

            else:
                # MAC wala part
                product = self.reg[q1] * self.reg[q2]
                result = self.reg[q3] + product

                self.C = 1 if result > 1 else 0
                self.reg[q3] = result % 2

                print(f"MAC → q{q3} = {self.reg[q3]}, Carry={self.C}")

                if self.reg[q3] == 1:
                    self.qc.x(q3)

                self.update_flags(self.reg[q3])

        self.last_used[q1] = self.pc-1


    def run(self):
        while True:
            instr = self.fetch()
            if instr is None:
                break

            op, q1, q2, q3 = self.decode(instr)
            self.execute(op, q1, q2, q3)

        for i in range(8):
            self.qc.measure(i, i)

        sim = Aer.get_backend('aer_simulator')
        compiled = transpile(self.qc, sim)
        result = sim.run(compiled, shots=1).result()

        print("\nFinal Output (8-bit):", result.get_counts())
        print("FLAGS → Z:", self.Z, " P:", self.P, " C:", self.C)


processor = QRISCProcessor()
processor.run()


[FETCH] PC=0 → 00000000
[DECODE] H | q1=0, q2=0, q3=0
[EXECUTE] H

[FETCH] PC=1 → 01010000
[DECODE] X | q1=1, q2=0, q3=0
[EXECUTE] X

[FETCH] PC=2 → 10000100
[DECODE] CX | q1=0, q2=1, q3=0
[EXECUTE] CX

[FETCH] PC=3 → 11011001
[DECODE] MAC/MEASURE | q1=1, q2=2, q3=1
[EXECUTE] MAC/MEASURE
MAC → q1 = 1, Carry=0

[FETCH] PC=4 → 11000000
[DECODE] MAC/MEASURE | q1=0, q2=0, q3=0
[EXECUTE] MAC/MEASURE
Measured q0

Final Output (8-bit): {'00000000': 1}
FLAGS → Z: 0  P: 1  C: 0
